In [13]:
import torch
from nnscf import denormalize
import os
import time

In [14]:
og_pt_dir = "./pt_dir/"
pt_dir = "./pt_dir_globalnorm/"
os.makedirs(og_pt_dir, exist_ok=True)
os.makedirs(pt_dir, exist_ok=True)

mod = "16qam" # Make sure this matches the data you want to fix!
# mod = "qpsk" # Make sure this matches the data you want to fix!

# clipping technique
# tech = "icf"
tech = "scf"

datasize = 100
# datasize = 120
# train_size = 80
train_size = 70 # should be the same as the one used in nnicf.ipynb, obviously
val_size = 10
test_size = 100 - train_size - val_size

In [15]:
print("--- STEP 1: Finding Global Limits from Training Data (Files 0 to train_size - 1) ---")
# Initialize global trackers
g_X_r_min, g_X_r_max = float('inf'), float('-inf')
g_Y_r_min, g_Y_r_max = float('inf'), float('-inf')
g_X_i_min, g_X_i_max = float('inf'), float('-inf')
g_Y_i_min, g_Y_i_max = float('inf'), float('-inf')

--- STEP 1: Finding Global Limits from Training Data (Files 0 to train_size - 1) ---


In [16]:
# Only scan the first train_size files to prevent Data Leakage!
for i in range(train_size):
    # Load dictionaries
    data_real = torch.load(os.path.join(og_pt_dir, f"{mod}_{tech}_part_{i:03d}_real.pt"), weights_only=True)
    data_imag = torch.load(os.path.join(og_pt_dir, f"{mod}_{tech}_part_{i:03d}_imag.pt"), weights_only=True)

    # Extract local limits and update globals
    g_X_r_min = min(g_X_r_min, data_real['X_min'])
    g_X_r_max = max(g_X_r_max, data_real['X_max'])
    g_Y_r_min = min(g_Y_r_min, data_real['Y_min'])
    g_Y_r_max = max(g_Y_r_max, data_real['Y_max'])

    g_X_i_min = min(g_X_i_min, data_imag['X_min'])
    g_X_i_max = max(g_X_i_max, data_imag['X_max'])
    g_Y_i_min = min(g_Y_i_min, data_imag['Y_min'])
    g_Y_i_max = max(g_Y_i_max, data_imag['Y_max'])
    print(f"Found Local Real X Limits: Min = {data_real['X_min']:.4f}, Max = {data_real['X_max']:.4f}")

print(f"Found Global Real X Limits: Min = {g_X_r_min:.4f}, Max = {g_X_r_max:.4f}")

Found Local Real X Limits: Min = -0.2293, Max = 0.2335
Found Local Real X Limits: Min = -0.2267, Max = 0.2288
Found Local Real X Limits: Min = -0.2342, Max = 0.2327
Found Local Real X Limits: Min = -0.2400, Max = 0.2441
Found Local Real X Limits: Min = -0.2306, Max = 0.2310
Found Local Real X Limits: Min = -0.2283, Max = 0.2245
Found Local Real X Limits: Min = -0.2276, Max = 0.2283
Found Local Real X Limits: Min = -0.2254, Max = 0.2404
Found Local Real X Limits: Min = -0.2348, Max = 0.2323
Found Local Real X Limits: Min = -0.2310, Max = 0.2216
Found Local Real X Limits: Min = -0.2186, Max = 0.2438
Found Local Real X Limits: Min = -0.2170, Max = 0.2326
Found Local Real X Limits: Min = -0.2351, Max = 0.2275
Found Local Real X Limits: Min = -0.2218, Max = 0.2549
Found Local Real X Limits: Min = -0.2521, Max = 0.2301
Found Local Real X Limits: Min = -0.2286, Max = 0.2177
Found Local Real X Limits: Min = -0.2312, Max = 0.2177
Found Local Real X Limits: Min = -0.2249, Max = 0.2587
Found Loca

In [17]:
print("\n--- STEP 2: Denormalizing and Renormalizing ALL 100 Files ---")
before_loop = time.time()
for i in range(100):
    loop_start = time.time() if i else before_loop

    real_path_og = os.path.join(og_pt_dir, f"{mod}_{tech}_part_{i:03d}_real.pt")
    imag_path_og = os.path.join(og_pt_dir, f"{mod}_{tech}_part_{i:03d}_imag.pt")

    data_real = torch.load(real_path_og, weights_only=True)
    data_imag = torch.load(imag_path_og, weights_only=True)

    # --- PROCESS REAL ---
    # 1. Denormalize back to raw physical voltages
    raw_X_r = denormalize(data_real['X_norm'], (data_real['X_min'], data_real['X_max']))
    raw_Y_r = denormalize(data_real['Y_norm'], (data_real['Y_min'], data_real['Y_max']))

    # todo: maybe i could make a function for that
    # 2. Renormalize using Global Limits
    data_real['X_norm'] = 2.0 * ((raw_X_r - g_X_r_min) / (g_X_r_max - g_X_r_min)) - 1.0 if g_X_r_max != g_X_r_min else raw_X_r
    data_real['Y_norm'] = 2.0 * ((raw_Y_r - g_Y_r_min) / (g_Y_r_max - g_Y_r_min)) - 1.0 if g_Y_r_max != g_Y_r_min else raw_Y_r

    # 3. OVERWRITE the old local limits with the global limits in the dictionary!
    data_real['X_min'], data_real['X_max'] = g_X_r_min, g_X_r_max
    data_real['Y_min'], data_real['Y_max'] = g_Y_r_min, g_Y_r_max

    # --- PROCESS IMAGINARY ---
    raw_X_i = denormalize(data_imag['X_norm'], (data_imag['X_min'], data_imag['X_max']))
    raw_Y_i = denormalize(data_imag['Y_norm'], (data_imag['Y_min'], data_imag['Y_max']))

    # todo: maybe i could make a function for that
    data_imag['X_norm'] = 2.0 * ((raw_X_i - g_X_i_min) / (g_X_i_max - g_X_i_min)) - 1.0 if g_X_i_max != g_X_i_min else raw_X_i
    data_imag['Y_norm'] = 2.0 * ((raw_Y_i - g_Y_i_min) / (g_Y_i_max - g_Y_i_min)) - 1.0 if g_Y_i_max != g_Y_i_min else raw_Y_i

    data_imag['X_min'], data_imag['X_max'] = g_X_i_min, g_X_i_max
    data_imag['Y_min'], data_imag['Y_max'] = g_Y_i_min, g_Y_i_max

    real_path_out = os.path.join(pt_dir, f"{mod}_{tech}_part_{i:03d}_real.pt")
    imag_path_out = os.path.join(pt_dir, f"{mod}_{tech}_part_{i:03d}_imag.pt")

    # Save files back to disk (overwriting)
    torch.save(data_real, real_path_out)
    torch.save(data_imag, imag_path_out)

    loop_end = time.time()
    print(f"Iteration {i + 1}/100 completed in {loop_end - loop_start:.2f} seconds")
    print(f"{loop_end - before_loop:.2f} seconds passed since before loop start")

print("Data successfully updated with Global Normalization!")


--- STEP 2: Denormalizing and Renormalizing ALL 100 Files ---
Iteration 1/100 completed in 0.54 seconds
0.54 seconds passed since before loop start
Iteration 2/100 completed in 0.49 seconds
1.04 seconds passed since before loop start
Iteration 3/100 completed in 0.51 seconds
1.55 seconds passed since before loop start
Iteration 4/100 completed in 0.50 seconds
2.05 seconds passed since before loop start
Iteration 5/100 completed in 0.52 seconds
2.57 seconds passed since before loop start
Iteration 6/100 completed in 0.52 seconds
3.09 seconds passed since before loop start
Iteration 7/100 completed in 0.55 seconds
3.64 seconds passed since before loop start
Iteration 8/100 completed in 0.55 seconds
4.18 seconds passed since before loop start
Iteration 9/100 completed in 0.53 seconds
4.71 seconds passed since before loop start
Iteration 10/100 completed in 0.52 seconds
5.23 seconds passed since before loop start
Iteration 11/100 completed in 0.51 seconds
5.74 seconds passed since before 

In [18]:
for i in range(train_size):
    # Load dictionaries
    data_real = torch.load(os.path.join(pt_dir, f"{mod}_{tech}_part_{i:03d}_real.pt"), weights_only=True)
    data_imag = torch.load(os.path.join(pt_dir, f"{mod}_{tech}_part_{i:03d}_imag.pt"), weights_only=True)

    # Extract local limits and update globals
    g_X_r_min = min(g_X_r_min, data_real['X_min'])
    g_X_r_max = max(g_X_r_max, data_real['X_max'])
    g_Y_r_min = min(g_Y_r_min, data_real['Y_min'])
    g_Y_r_max = max(g_Y_r_max, data_real['Y_max'])

    g_X_i_min = min(g_X_i_min, data_imag['X_min'])
    g_X_i_max = max(g_X_i_max, data_imag['X_max'])
    g_Y_i_min = min(g_Y_i_min, data_imag['Y_min'])
    g_Y_i_max = max(g_Y_i_max, data_imag['Y_max'])
    print(f"Found Local Real X Limits: Min = {g_X_r_min:.4f}, Max = {g_X_r_max:.4f}")

print(f"Found Global Real X Limits: Min = {g_X_r_min:.4f}, Max = {g_X_r_max:.4f}")

Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Local Real X Limits: Min = -0.2798, Max = 0.2718
Found Loca